# Re-run the Salary and Founding Year experiments on a Colab T4

Open this notebook in Google Colab, select **Runtime > Change runtime type > T4 GPU**, and choose **Run all**.

- The notebook embeds the 56 protocol-locked files of both experiments (inputs, scripts, configuration), checks each against the SHA-256 in its `protocol_lock.json`, and stops if any differs.
- It then runs the unmodified scripts for each task: `infer --split dev` → `infer --split main` → `analyze`.
- Model: `Qwen/Qwen2.5-1.5B-Instruct@989aa79`, FP16, greedy decoding; no training.
- Libraries: transformers 4.57.6, tokenizers 0.22.2. Before generation, the notebook checks that all 14,256 inputs have the locked token counts.
- The main split took about 45 minutes per task on a T4. If the development split projects more than two hours for the main split, the main split is not run (a rule fixed before the experiment).
- When a task finishes, a results ZIP is downloaded. Set `USE_DRIVE = True` below to keep progress on Google Drive and resume after a disconnect; outputs that already exist are never regenerated.
- Files are restored under `/home/cv/SSD0/jang/nlp/experiments/`, the path that the locked configuration files point to.

In [ ]:
# ---- 설정 ----
USE_DRIVE = False                              # True: 진행 상황을 Google Drive에 저장(끊겨도 이어서 실행)
DRIVE_DIR = "/content/drive/MyDrive/hw2_t4_rerun"
RUN_TASKS = ["salary", "year"]                 # 한 과제만 돌리려면 하나만 남기세요

In [ ]:
import platform, subprocess, sys
from pathlib import Path
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("Colab 런타임에서만 실행하세요.")
import google.colab  # noqa: F401  (Colab이 아니면 여기서 실패)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout.strip()
print("GPU:", gpu)
if "T4" not in gpu:
    raise RuntimeError("T4 GPU가 아닙니다. 런타임 > 런타임 유형 변경 > T4 GPU를 선택하세요.")
print("Python:", sys.version.split()[0])

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.6", "tokenizers==0.22.2",
                "accelerate>=1.0,<2", "safetensors>=0.4,<1"], check=True)
print(subprocess.run([sys.executable, "-c",
      "import torch, transformers, tokenizers, numpy, matplotlib; "
      "print('torch', torch.__version__, 'cuda', torch.version.cuda); "
      "print('transformers', transformers.__version__, 'tokenizers', tokenizers.__version__); "
      "print('numpy', numpy.__version__, 'matplotlib', matplotlib.__version__)"],
      capture_output=True, text=True, check=True).stdout)

In [ ]:
import base64, hashlib, io, json, os, shutil, tarfile
from pathlib import Path
BASE = Path("/home/cv/SSD0/jang/nlp")          # 원래 경로 그대로(설정 파일의 cache_dir와 모듈 경로가 이 위치를 가정)
EXP = BASE / "experiments"
TASKS = {"salary": {"dir": "hw2_missing_fields", "run_id": "run_20261031_hw1_main_v2"}, "year": {"dir": "hw2_year_generalization", "run_id": "run_20261102_founding_year"}}
LOCK_HASHES = {"salary": {"audit/protocol_v1_before_currency_format_fix.zip": "ddb191a4a07ba213ed2699ca226676ba87ffbdfe61e9da3cc463d4721f1052d1", "config.json": "8870bfed5439bfd0eab47da871828baccb0d705eef47269c6cc4a310291f7cc9", "data/backgrounds.jsonl": "a1133f17de00150bb3fa3c73336bd6a1cfa5c4cc8bea39eb0a62578de358444f", "data/data_audit.json": "69b86931c5ce8af95523a0ed357b46606a7d01ce7fe9c9717a4e542958c1cd3c", "data/dev_bases.jsonl": "d022ddf82016faa97066623773ecf734554c8d84d1111c753cd6bc33a2242307", "data/dev_requests.jsonl": "5752258b1aef3a510d41947d94ce703bd92396deb8174bc89d3f4145886ea538", "data/main_bases.jsonl": "e9b636fa4ec666c6377cc1bca53ddeec6a8e866d146429899501dcc3fc0fe278", "data/main_requests.jsonl": "ccaf3e10d27da63d16302b244f6e9c721d6270ee22a28b88c84a60bbad640966", "data/static_checks.json": "b6fb2bf429af210dd6186ea6966d69716d998972c910b9ade11d3c46a1f7dd52", "experiment_spec.md": "714c41a51ace3d97060bb5a63d2605a155a7cf844440f9248b1b543a67419abd", "requirements_frozen.txt": "b5257ddd5482206274949a6d4f202b39bc14760cd7ca09b3996022bc5d11e740", "scripts/__init__.py": "1c2a46407a9ed879e8631d10da669c18de755fa1b4b317cbd30c7faff82649c6", "scripts/analyze.py": "34989edd3e44583eadfebe40c97b54053b6a11c7c643cc3874a19a37175180e8", "scripts/check_and_lock.py": "38e7fba3013ec0390184ec311e14d4a12d7dbb0534017d91365caebfc91fb9d9", "scripts/common.py": "4bb45fd6e41bce333c51a061f60547152d869fee76fd6d585474e10db5d959d0", "scripts/infer.py": "44e60865ea8240971e6f892c0901640e2a6fa0bbcaef41ab6a06d33c1d2f1584", "scripts/prepare.py": "342c3e3c2e3753a2f68c1f714f467da375717e03902d1b6c55e8d82389bbfa53", "source/alice_gutenberg_11.txt": "01b38ea4c710a84bc18d0bd41271a5a1a92b94e97b2812f4dece97d4a694725e", "source/design_references.json": "500ac2aaae99304cc736d1f9545eda47174188f924803ff6efe95f322ebd56a5", "source/filtered_paragraphs.jsonl": "eaff3044a52fc59e9699f30dece28eea81dee60c35b6cb859e98a90ed20337ba", "source/source_manifest.json": "97650d1c4d93bf448fbdf0a0cc333db52cdd1ccadfbd07dfd57294b41585baee"}, "year": {"config.json": "ab64d31b12601b9e6c6a09d17d69608c6e0403122b45e7decc6cbfe1cc13cf95", "data/backgrounds.jsonl": "8380f811281abf152cbef66b41b39ccbd96be5524a0e6c454a027ab70c62fa70", "data/data_audit.json": "33f1112a640bba83ff0451f3419793a641a352f8b64e34a08f385c17350312ca", "data/dev_bases.jsonl": "07098db384d8e88e1c1ca0b020f037fc06f5c770e3a8893e4ef53bccd61999c2", "data/dev_requests.jsonl": "85c45d6977efe59e0f766bd85254e0da72a8d68b32451a8d4f5f6359ad851025", "data/main_bases.jsonl": "ebe88657843f2c0b96c8f43186b77108b9dbb1db3efe3034fee799bf7bba3b3a", "data/main_requests.jsonl": "67fda69c9983c8d8c27d2ab2cc33d6577f30c69fc45ee0786c107c785ffea221", "data/static_checks.json": "eb22f48d365d09ccb79cb9e7b9790f1afc97d24ff1ab7828a0a5c8b021b51e86", "experiment_spec.md": "8438ecff3728e51a853f5ba1409e3147358aa581ba5e12b4f453f79aa40d9ea1", "reference/salary_completion_manifest.json": "e2b2e3e8d0306cbeb182cb7d2d52ca5c766f7db7512ce0237e5328ab3ee0b46a", "reference/salary_config.json": "8870bfed5439bfd0eab47da871828baccb0d705eef47269c6cc4a310291f7cc9", "reference/salary_contrasts.csv": "1ffcadd27c9e67819cfa77714aa068ee0e14cd29bed8d113c58f5aa7d6646b44", "reference/salary_notation_sensitivity.csv": "45313cee71b7d1244d9124f9c61998d6bd7c62491706109aab4a855e5e775a69", "reference/salary_report_ko.md": "f3c4cc3a9cf0aa6cfe0b97cb7f78221ac71aec3d115626a790f5dc14ce3b0eac", "reference/salary_summary.json": "1625559701b2aec1c1dae6265185d3bfcbed070577ff2ad6f7198acfa19e133e", "requirements_frozen.txt": "b5257ddd5482206274949a6d4f202b39bc14760cd7ca09b3996022bc5d11e740", "requirements_minimal.txt": "5b26ab7e47641b6d843ee6d87f231672b9f2ba3099dc61ab06a87b8d8d3372c0", "scripts/__init__.py": "70a96fba812865e19a4380bd9702d779d95279f0bec99d48e628259e78a50dfe", "scripts/analyze.py": "f07ac0c282caed56411f2a2cfafb2d0c0e36fd12d090dfd3b6521505134a64cb", "scripts/check_and_lock.py": "035b54d51113160b9a67a4f4c8738c8e6f2af6bc6fe58291ce952d55caf02aef", "scripts/common.py": "39546a2f13de633b88b35bd6e1fc154a98043a7f9c1aea32cd1b58130b6fb0bb", "scripts/infer.py": "58a22de6b29c97c688bdd4f3a458f352c80aa292a9b7c284d8c4442d69535808", "scripts/prepare.py": "9bfeaeb30c8859fa19d5cd2b7a627c9d4b851f2113114860051e18be199aee29", "scripts/statistics.py": "aca4383ef4af2d843b8b665d9b35d5bd6a21e524877821a514063c4699552a72", "source/alice_gutenberg_11.txt": "01b38ea4c710a84bc18d0bd41271a5a1a92b94e97b2812f4dece97d4a694725e", "source/design_references.json": "500ac2aaae99304cc736d1f9545eda47174188f924803ff6efe95f322ebd56a5", "source/filtered_paragraphs.jsonl": "eaff3044a52fc59e9699f30dece28eea81dee60c35b6cb859e98a90ed20337ba", "source/original_download_manifest.json": "97650d1c4d93bf448fbdf0a0cc333db52cdd1ccadfbd07dfd57294b41585baee", "source/source_manifest.json": "21b65648189b219d08c8af55a08f00ce2801ab16d217e9db7df88dcd79e36a20"}}
BUNDLE_SHA = "f4d78a2cc611fc3575356f012326bc5e11e15a5724491d6267f7d3e9fc5c6e4c"
BUNDLE = base64.b64decode("...")
if hashlib.sha256(BUNDLE).hexdigest() != BUNDLE_SHA:
    raise RuntimeError("내장 파일 묶음이 손상되었습니다")
sha = lambda b: hashlib.sha256(b).hexdigest()
with tarfile.open(fileobj=io.BytesIO(BUNDLE), mode="r:xz") as tf:
    for member in tf.getmembers():
        rel = Path(member.name)
        if rel.is_absolute() or ".." in rel.parts or not member.isfile():
            raise RuntimeError("허용되지 않는 경로: " + member.name)
        data = tf.extractfile(member).read()
        dest = EXP / rel
        if dest.exists():
            if sha(dest.read_bytes()) != sha(data):
                raise RuntimeError("이미 있는 파일이 내장본과 다릅니다(덮어쓰지 않음): " + str(dest))
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
for key, t in TASKS.items():
    root = EXP / t["dir"]
    run = root / "runs" / t["run_id"]
    if USE_DRIVE and not run.is_symlink():
        target = Path(DRIVE_DIR) / t["dir"] / t["run_id"]
        target.mkdir(parents=True, exist_ok=True)
        lock = run / "protocol_lock.json"
        if (target / "protocol_lock.json").exists():
            if sha((target / "protocol_lock.json").read_bytes()) != sha(lock.read_bytes()):
                raise RuntimeError("Drive의 protocol_lock이 다릅니다: " + str(target))
        else:
            shutil.copy2(lock, target / "protocol_lock.json")
        shutil.rmtree(run)
        run.symlink_to(target, target_is_directory=True)
    bad = [p for p, h in LOCK_HASHES[key].items() if sha((root / p).read_bytes()) != h]
    if bad:
        raise RuntimeError({key: bad})
    # analyze.py saves plots to figures/ without creating it; the tar bundle has no empty folders.
    (root / "figures").mkdir(exist_ok=True)
    print(key, "| 잠긴 파일", len(LOCK_HASHES[key]), "개 일치 | run 폴더:", run.resolve(),
          "| 기존 출력:", sorted(p.name for p in run.resolve().iterdir()))

In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
                         cache_dir="/home/cv/SSD0/jang/nlp/.cache/huggingface/hub")
if not path.rstrip("/").endswith("989aa7980e4cf806f80c7fef2b1adb7bc71aa306"):
    raise RuntimeError("모델 revision 불일치: " + path)
print("모델 준비 완료:", path)

In [ ]:
# 전체 입력 14,256개의 토큰 수가 잠긴 값과 같은지 먼저 확인(다르면 생성 전에 중단)
import subprocess, sys
CHECK = r'''
import json, sys
sys.path.insert(0, "/home/cv/SSD0/jang/nlp")
for d in ("hw2_missing_fields", "hw2_year_generalization"):
    common = __import__(f"experiments.{d}.scripts.common", fromlist=["x"])
    tok = common.tokenizer()
    for split in ("dev", "main"):
        rows = common.read_jsonl(common.ROOT / f"data/{split}_requests.jsonl")
        enc = tok([common.render(r["messages"], tok) for r in rows], add_special_tokens=False)["input_ids"]
        bad = [r["request_id"] for r, ids in zip(rows, enc) if len(ids) != r["input_token_count"]]
        print(d, split, len(rows), "inputs, token-count mismatches:", len(bad), bad[:3], flush=True)
        if bad:
            sys.exit(1)
'''
r = subprocess.run([sys.executable, "-c", CHECK], cwd="/home/cv/SSD0/jang/nlp", text=True,
                   capture_output=True)
print(r.stdout); print(r.stderr[-2000:] if r.returncode else "")
if r.returncode:
    raise RuntimeError("토큰 수가 달라졌습니다. 실행하지 말고 결과를 공유해 주세요.")

In [ ]:
import csv, json, subprocess, sys, zipfile
from pathlib import Path

def run(module, *args):
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=str(BASE), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"실패: {module} {args} (출력은 run 폴더에 보존됨)")

def package(key):
    t = TASKS[key]; root = EXP / t["dir"]; run_dir = (root / "runs" / t["run_id"]).resolve()
    out = Path(f"/content/hw2_t4_{key}_results.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for folder, arc in ((run_dir, f"runs/{t['run_id']}"), (root / "results", "results"), (root / "figures", "figures")):
            for p in sorted(folder.rglob("*")):
                if p.is_file():
                    z.write(p, f"{t['dir']}/{arc}/{p.relative_to(folder)}")
        for name in ("final_report_ko.md", "completion_manifest.json"):
            if (root / name).exists():
                z.write(root / name, f"{t['dir']}/{name}")
    print("결과 ZIP:", out, round(out.stat().st_size / 1e6, 1), "MB")
    if USE_DRIVE:
        import shutil; shutil.copy2(out, Path(DRIVE_DIR) / out.name)
    from google.colab import files
    files.download(str(out))

def finished(key, split):
    # A finished split must not be re-run: infer.py would overwrite {split}_runtime.json
    # (and pilot_audit.json for dev) with the timings of a run that generated nothing.
    t = TASKS[key]; p = (EXP / t["dir"] / "runs" / t["run_id"]).resolve() / f"{split}_runtime.json"
    if not p.exists():
        return False
    rt = json.loads(p.read_text())
    return rt.get("status") == "COMPLETE" and rt.get("completed") == rt.get("requests")

def run_task(key):
    module = f"experiments.{TASKS[key]['dir']}.scripts"
    for split in ("dev", "main"):
        if finished(key, split):
            print(f"{key} {split}: 이미 완료됨 → 추론을 다시 실행하지 않음(실행 시간 기록 보존)", flush=True)
        else:
            run(module + ".infer", "--split", split, "--device", "cuda:0")
    run(module + ".analyze")
    package(key)

## Salary (dev → main → analyze → ZIP download)

In [ ]:
if "salary" in RUN_TASKS:
    run_task("salary")

## Founding Year (dev → main → analyze → ZIP download)

In [ ]:
if "year" in RUN_TASKS:
    run_task("year")

## Summary

In [ ]:
# 요약: 주요 대비(%p, 95% CI)와 실행 환경
for key in RUN_TASKS:
    root = EXP / TASKS[key]["dir"]
    rows = list(csv.DictReader((root / "results/contrasts.csv").open()))
    rows = [r for r in rows if r["subset"] == "ALL" and r.get("policy", "STRICT") == "STRICT"
            and r["metric"] in ("delta_attr", "delta_C", "delta_U", "delta_type")]
    rt = json.loads(((root / "runs" / TASKS[key]["run_id"]).resolve() / "main_runtime.json").read_text())
    print(f"== {key} | GPU {rt['gpu']} | main {rt['process_seconds']/60:.1f} min | valid {rt['strict_valid']}/{rt['requests']}")
    for r in rows:
        print(f"  {r['metric']:<11} {100*float(r['estimate']):+7.2f}  [{100*float(r['ci95_low']):+.2f}, {100*float(r['ci95_high']):+.2f}]")